In [31]:
import os
from local.utils import *



def get_anno_datas(anno_dir, wav_scp):
    
    json_files = get_dir_files(anno_dir, '.json')
    print(f'get {len(json_files)} json files from {anno_dir}')


    anno_datas = []
    for json_file in json_files:
        utt = os.path.basename(json_file).replace('.json', '').replace('_mp3', '')
        if utt not in wav_scp:
            print(f'waring {utt} not in wav_scp')
            continue
        
        name = utt.replace('@10.190.101.161_0', '_ch0')
        name = name.replace('@10.190.101.161_1', '_ch1')
        anno_data = read_json_data(json_file)
        anno_data['utt'] = name
        anno_data['wav'] = wav_scp[utt]
        anno_datas.append(anno_data)

    print(f'total get {len(anno_datas)} data')
    
    return anno_datas
    
    

wav_dir = '/data/nas/dataset/asr/kefu/huaian3/select_for_biaobei/'
anno_dir = '/data/nas/dataset/asr/kefu/huaian3/annos/mid_anno/'

anno_checked_dir = '/data/nas/dataset/asr/kefu/huaian3/annos/anno_checked/'
anno_no_checked_dir = '/data/nas/dataset/asr/kefu/huaian3/annos/anno_no_checked/'


wav_scp = get_wav_scp(wav_dir, '.wav')
print(f'get {len(wav_scp.keys())} wav files')


checked_anno_datas = get_anno_datas(anno_checked_dir, wav_scp)
no_checked_anno_datas = get_anno_datas(anno_no_checked_dir, wav_scp)



get 7000 wav files
get 1000 json files from /data/nas/dataset/asr/kefu/huaian3/annos/anno_checked/
total get 1000 data
get 1000 json files from /data/nas/dataset/asr/kefu/huaian3/annos/anno_no_checked/
total get 1000 data


In [33]:
import os
import soundfile as sf
import tqdm
# def merge_time_stamps(anno):
    
#     segs_anno = anno['segs']
    

def split_audio_by_timestamps(anno, out_dir):
    utt = anno['utt']
    wav_path = anno['wav']
    if not os.path.exists(wav_path):
        return
    speech, fs = sf.read(wav_path)
    num_sample_ms = int(fs/1000)
    
    dur = 0
    for i, seg in enumerate(anno['segs']):
        st, ed = seg['seg']
        txt = remove_punctuation(seg['text'])
        # print(f'[{st}, {ed}]: {txt}')
        seg_speech = speech[int(st*num_sample_ms):int(ed*num_sample_ms)]
        seg_utt = f'{out_dir}/{utt}_seg{i:03d}'
        sf.write(f'{seg_utt}.wav', seg_speech, fs)
        with open(f'{seg_utt}.txt', 'w') as f:
            f.write(f'{txt}\n')
        dur += ed - st
    
    dur = dur/1000
    
    return dur, len(speech)/fs


def split_audio_by_timestamps_batch(annos, out_seg_wav_dir):
    total_seg_dur = 0
    total_speech_dur = 0
    for wav_anno in tqdm.tqdm(annos):
        seg_dur, speech_dur = split_audio_by_timestamps(wav_anno, out_seg_wav_dir)
        total_seg_dur += seg_dur
        total_speech_dur += speech_dur

    print(f'total seg dur: {total_seg_dur} speech_dur {total_speech_dur}')
    
    return total_seg_dur, total_speech_dur
        


In [35]:
import tqdm

out_seg_wav_dir = '/data/nas/dataset/asr-train/kefu_train/huaian_biaobei_data01/wavs/'

out_seg_wav_dir_checked = '/data/nas/dataset/asr-train/kefu_train/huaian_biaobei_data01/data_checked/wavs/'
out_seg_wav_dir_no_checked = '/data/nas/dataset/asr-train/kefu_train/huaian_biaobei_data01/data_no_checked/wavs/'

total_seg_dur, total_speech_dur = split_audio_by_timestamps_batch(checked_anno_datas, out_seg_wav_dir_checked)
total_seg_dur, total_speech_dur = split_audio_by_timestamps_batch(no_checked_anno_datas, out_seg_wav_dir_no_checked)


100%|██████████| 1000/1000 [04:30<00:00,  3.69it/s]


total seg dur: 36318.96899999995 speech_dur 108071.99999999996


100%|██████████| 1000/1000 [04:14<00:00,  3.94it/s]

total seg dur: 35229.00700000002 speech_dur 107760.39999999989


数据集划分

In [ ]:
data_dir = '/data/nas/dataset/asr-train/kefu_train/huaian_biaobei_data01/data_checked'
text_file = f'{data_dir}/text'
wav_scp_file = f'{data_dir}/wav.scp'



In [ ]:
print(anno_datas[0])
print(anno_datas[0].keys())
print(anno_datas[0]['utt'], anno_datas[0]['wav'])

out_seg_wav_dir = '/data/nas/dataset/asr-train/kefu_train/huaian_biaobei_data01/wavs/'

for seg in anno_datas[0]['segs']:
    time_stamp = seg['seg']
    txt = seg['text']
    txt = remove_punctuation(txt)
    print(time_stamp, txt)
print(anno_datas[0]['full_text'])

seg_dur, speech_dur = split_audio_by_timestamps(anno_datas[0], out_seg_wav_dir)
print(seg_dur, speech_dur)